# non-diff-fn-wrap — ex1: wrap a non-differentiable op (eq) — no Recipe, no requires_grad

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `non-diff-fn-wrap`. Running the final beacon cell reports progress against the `Backprop: non-differentiable fn wrap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: non-differentiable fn wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`non-diff-fn-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "non-diff-fn-wrap"
DD_SUBTOPIC = "Backprop: non-differentiable fn wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## non-differentiable fn wrap — quick refresher

Some forward ops have no useful gradient — e.g. `torch.eq` returns a bool, `torch.argmax` returns indices. We still want to call them on `MiniTensor`s, but we should NOT build a Recipe or set `requires_grad=True` on the output.

Solution: a per-op flag plumbed into the wrapper at register time:

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        ...
        requires_grad = (
            grad_tracking_enabled
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            out.recipe = Recipe(...)   # only built when grad will flow
        return out
    return tensor_func

eq = wrap_forward_fn(torch.eq, is_differentiable=False)
```

Two effects:
- Output of non-diff op is ALWAYS `requires_grad=False`, even if all inputs are tracked.
- No `Recipe` is attached, so reverse pass treats the output as a leaf — the graph terminates here naturally.

### Exercise 1 — wrap a non-differentiable op (eq) — no Recipe, no requires_grad

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the non-differentiable-op wrap path: when is_differentiable is False, return a MiniTensor whose requires_grad is False and whose recipe is None, regardless of the inputs.
> Keywords: non-differentiable, eq, argmax, no-recipe, wrap
> ```

**KCs targeted:** `non-diff-fn-wrap`, `is-differentiable-flag`

Implement `wrap_forward_fn(fwd_fn, is_differentiable=True)`. It must:

1. Unbox MiniTensor inputs to their raw `.array`s (pass non-Tensors through unchanged).
2. Call `fwd_fn(*raw_args, **kwargs)`.
3. Box the result as a `MiniTensor`. Compute `requires_grad` via the **three-gate AND**: `grad_tracking_enabled AND is_differentiable AND any(input is a tracked MiniTensor)`.
4. **Only if `requires_grad` is True**, attach a `Recipe`. Otherwise leave `recipe = None`.

**Why the conditional Recipe matters.** For `eq = wrap_forward_fn(torch.eq, is_differentiable=False)`, the output is a `bool` tensor — there is no gradient to compute, and we don't want the reverse pass to try to traverse `eq`'s parents. Setting `recipe = None` makes the output behave like a leaf in the computational graph — backprop stops here.

The setup cell provides `MiniTensor`, `Recipe`, and the module-level `grad_tracking_enabled = True`. After you implement `wrap_forward_fn`, the test wraps THREE ops:
- `add = wrap_forward_fn(torch.add)` — differentiable.
- `eq = wrap_forward_fn(torch.eq, is_differentiable=False)` —   not differentiable.
- `argmax = wrap_forward_fn(torch.argmax, is_differentiable=False)` —   not differentiable.

And checks the requires_grad / recipe state on each.

Don't call `torch.autograd`.

In [ ]:
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        # 1. unbox
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # 2. forward call
        out_arr = fwd_fn(*raw_args, **kwargs)
        # 3. three-gate AND (read the global toggle FRESH each call)
        requires_grad = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(
                isinstance(a, MiniTensor) and a.requires_grad for a in args
            )
        )
        out = MiniTensor(out_arr, requires_grad)
        # 4. conditional Recipe — only when grad will flow
        if requires_grad:
            parents = {
                idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)
            }
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func


<details><summary>Solution</summary>

```python
def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        # 1. unbox
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        # 2. forward call
        out_arr = fwd_fn(*raw_args, **kwargs)
        # 3. three-gate AND (read the global toggle FRESH each call)
        requires_grad = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(
                isinstance(a, MiniTensor) and a.requires_grad for a in args
            )
        )
        out = MiniTensor(out_arr, requires_grad)
        # 4. conditional Recipe — only when grad will flow
        if requires_grad:
            parents = {
                idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)
            }
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

**Why skip Recipe entirely (vs. building it with `requires_grad=False`).** A node with `recipe=None` is treated by the reverse pass as a leaf — `sorted_computational_graph` stops there. If you build the Recipe anyway, the traversal continues past it, walks into non-differentiable ancestors, and either crashes (no back fn registered for `torch.eq`) or wastes work.

**`is_differentiable` is a CLOSURE-captured constant.** `wrap_forward_fn(torch.eq, is_differentiable=False)` returns a `tensor_func` whose closure remembers `False`. Every subsequent call short-circuits on gate 2. You CANNOT change this at runtime without re-wrapping the op — which is the point: differentiability is a property of the op, not the call site.

**Why read `grad_tracking_enabled` via `globals()`.** A naive bare reference closes over the value at function-definition time. Re-binding the module-level name (e.g. via a `NoGrad` context manager) wouldn't change the closure's value. `globals()['...']` always reads the current binding.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()